In [4]:
from pyspark.sql import SparkSession, Row

spark = SparkSession.builder.appName('SparkSQL').getOrCreate()

data = spark.read.csv('fakefriendheader.csv', header=True, inferSchema=True)

data.createOrReplaceTempView('fake_friends')


data.show()

data.select('name', 'age').show()

data.groupBy('age').avg('#friends').sort('age').show()

data.printSchema()

spark.stop()

+---+--------+---+--------+
| id|    name|age|#friends|
+---+--------+---+--------+
|  0|    Will| 33|     385|
|  1|Jean-Luc| 33|       2|
|  2|    Hugh| 55|     221|
|  3|  Deanna| 40|     465|
|  4|   Quark| 68|      21|
|  5|  Wesley| 17|       1|
+---+--------+---+--------+

+--------+---+
|    name|age|
+--------+---+
|    Will| 33|
|Jean-Luc| 33|
|    Hugh| 55|
|  Deanna| 40|
|   Quark| 68|
|  Wesley| 17|
+--------+---+

+---+-------------+
|age|avg(#friends)|
+---+-------------+
| 17|          1.0|
| 33|        193.5|
| 40|        465.0|
| 55|        221.0|
| 68|         21.0|
+---+-------------+

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- #friends: integer (nullable = true)



In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName('UDF-EG').getOrCreate()

@udf(returnType=StringType())
def user_details(name, age):
	return f"Name: {name}, Age: {age}"

people = spark.read.csv('fakefriendheader.csv', header=True)

people.withColumn("name_age", user_details('name', 'age')).show()

spark.stop()

+---+--------+---+--------+--------------------+
| id|    name|age|#friends|            name_age|
+---+--------+---+--------+--------------------+
|  0|    Will| 33|     385| Name: Will, Age: 33|
|  1|Jean-Luc| 33|       2|Name: Jean-Luc, A...|
|  2|    Hugh| 55|     221| Name: Hugh, Age: 55|
|  3|  Deanna| 40|     465|Name: Deanna, Age...|
|  4|   Quark| 68|      21|Name: Quark, Age: 68|
|  5|  Wesley| 17|       1|Name: Wesley, Age...|
+---+--------+---+--------+--------------------+



In [12]:
from pyspark.sql import SparkSession, functions as func

spark = SparkSession.builder.appName('Word Count').getOrCreate()

df = spark.read.text('pride_and_prejudice.txt')
words = df.select(func.explode(func.split(df.value, r'\W+')).alias('word'))
wordsNoEmpty = words.filter(words.word != '')
lowercaseWords = wordsNoEmpty.select(func.lower(wordsNoEmpty.word).alias('word'))

lowercaseWords.groupBy('word').count().sort('count', ascending=False).show()


#df.show()

spark.stop()


+----+-----+
|word|count|
+----+-----+
| the| 4846|
|  to| 4405|
|  of| 3962|
| and| 3835|
| her| 2260|
|   i| 2098|
|   a| 2094|
|  in| 2051|
| was| 1874|
| she| 1732|
|that| 1620|
|  it| 1603|
| not| 1520|
| you| 1417|
|  he| 1350|
| his| 1289|
|  be| 1280|
|  as| 1239|
| had| 1181|
|with| 1149|
+----+-----+
only showing top 20 rows


In [15]:
from pyspark.sql import SparkSession, functions as func 
from pyspark.sql.types import FloatType, IntegerType, StringType, StructType, StructField

spark = SparkSession.builder.appName('Example').getOrCreate()

schema = StructType([
	StructField('StationID', StringType(), True),
	StructField('Date', IntegerType()),
	StructField('EntryType', StringType()),
	StructField('Value', FloatType())
])

df = spark.read.csv(
	'weatherData1800s.csv',
	schema=schema,
	header=False
)

df.printSchema()

minTemps = df.filter(df.EntryType == 'TMIN')

stationTemps = minTemps.select('StationID', 'Value')

minStationTemp = stationTemps.groupBy('StationID').min('Value').show()


spark.stop()


root
 |-- StationID: string (nullable = true)
 |-- Date: integer (nullable = true)
 |-- EntryType: string (nullable = true)
 |-- Value: float (nullable = true)

+-----------+----------+
|  StationID|min(Value)|
+-----------+----------+
|ITE00100554|    -148.0|
|EZE00100082|    -135.0|
+-----------+----------+



# Challenge

In [29]:
from pyspark.sql import SparkSession, functions as func 
from pyspark.sql.types import FloatType, IntegerType, StringType, StructType, StructField

spark = SparkSession.builder.appName('Customer Orders').getOrCreate()

schema = StructType([
	StructField('CustomerID', IntegerType()),
	StructField('OrderID', IntegerType()),
	StructField('OrderValue', FloatType())
])

df = spark.read.csv(
	'customer-orders.csv',
	schema=schema,
	header=False
)

df.printSchema()

customerTotals = df.select('CustomerID', 'OrderValue').groupBy('CustomerID').agg(func.sum('OrderValue').alias('Totals')).sort('Totals', ascending=False)

customerTotals.show()

spark.stop()

root
 |-- CustomerID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- OrderValue: float (nullable = true)

+----------+------------------+
|CustomerID|            Totals|
+----------+------------------+
|        68| 6375.450028181076|
|        73| 6206.199985742569|
|        39| 6193.109993815422|
|        54| 6065.390002984554|
|        71| 5995.659991919994|
|         2| 5994.589979887009|
|        97| 5977.190007060766|
|        46| 5963.110011339188|
|        42| 5696.840004444122|
|        59| 5642.890004396439|
|        41| 5637.619991332293|
|         0| 5524.950008839369|
|         8|5517.2399980425835|
|        85|  5503.42998456955|
|        61| 5497.479998707771|
|        32| 5496.049998283386|
|        58| 5437.730004191399|
|        63| 5415.150004655123|
|        15| 5413.510010659695|
|         6| 5397.880012750626|
+----------+------------------+
only showing top 20 rows


In [3]:
from pyspark.sql import SparkSession
import os
import pyspark.pandas as ps

os.environ['PYARROW_IGNORE_TIMEZONE'] = "1"

spark = SparkSession.builder \
    .appName("Sparky Panda") \
    .config("spark.sql.ansi.enable", "false") \
    .config("spark.executorEnv.PYARROW_IGNORE_TIMEZONE", "1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
ps_df = ps.DataFrame({
    "id": [1,2,3,4,5],
    "name": ["Alice", "Bobby", "David", "Charlie", "Jimmy"],
    "age": [25, 30, 28, 96, 32],
    "salary": [50000, 60000, 87000, 123000, 64000]
})

print(ps_df)

ps_df['promo_salary'] = ps_df['salary'] * 1.1
ps_df.to_spark().show()


spark.stop()

   id     name  age  salary
0   1    Alice   25   50000
1   2    Bobby   30   60000
2   3    David   28   87000
3   4  Charlie   96  123000
4   5    Jimmy   32   64000


/home/aldn/data_eng/spark/.venv/lib/python3.12/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


+---+-------+---+------+-----------------+
| id|   name|age|salary|     promo_salary|
+---+-------+---+------+-----------------+
|  1|  Alice| 25| 50000|55000.00000000001|
|  2|  Bobby| 30| 60000|          66000.0|
|  3|  David| 28| 87000|95700.00000000001|
|  4|Charlie| 96|123000|         135300.0|
|  5|  Jimmy| 32| 64000|          70400.0|
+---+-------+---+------+-----------------+

